# Carvana: daily inventory and page-status checks

Read this notebook in order: **population and time -> inventory -> VIN changes ->
page evidence -> next step**. Run All only reads retained files and SQLite; it makes
no requests, imports, or exports. [Notebook 10](10_carvana_inventory.ipynb) explains
the POST request. [Notebook 21](21_carvana_intraday_reference.ipynb) preserves the
older intraday source-to-SQL experiment and coverage audits.

Our pilot is Tesla Model 3, model years 2020-2026, seven fixed year queries, ZIP
08542, location filtering disabled. Completeness means this configured population
passed its checks, not national Carvana inventory.

The first daily snapshot is September 8, 2026 at about 21:45 New York time:
674 VINs and 215 native pending flags. The page study later that same evening
checked five of those listings. It did **not** create a second daily snapshot.
The tables below read the evidence available at the displayed cutoff.

```text
POST search pages -> saved responses -> registered daily cycle -> SQLite inventory
                                                 |
                                compare retailer + VIN across days
                                                 |
                           check selected vehicle pages -> save exact wording
```

VIN identifies the vehicle; listing ID identifies its listing. We keep both.
Pending, disappearance, an explicit Sold label, and a completed sale are different
observations. Asking prices are USD, mileage is miles, source clocks are UTC,
and inventory dates use America/New_York.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if (ROOT / 'vehicle/src').is_dir():
    ROOT = ROOT / 'vehicle'
elif ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

In [2]:
from vehicle_tracker.daily import (tracking_settings, tracking_history, tracking_report, daily_tables,
                                   registered_cycles, review_inputs)
from vehicle_tracker.cycles import read_cycle_history
from vehicle_tracker.checks import CHECK_COLUMNS, select_reviews, read_records, validate_checks
from vehicle_tracker.sales import REVIEW_COLUMNS

TRACKING_CONFIG_PATH = Path(globals().get('TRACKING_CONFIG_OVERRIDE', ROOT / 'config/carvana_daily_tracking.json'))
TRACKING_AS_OF = globals().get('AS_OF_OVERRIDE', pd.Timestamp.now(tz='UTC').isoformat())
tracking_tables = {}
if globals().get('RUN_TRACKING_VIEW_OVERRIDE', True):
    tracking = tracking_settings(TRACKING_CONFIG_PATH)
    check_override = pd.DataFrame(CHECKS_OVERRIDE, columns=CHECK_COLUMNS) if 'CHECKS_OVERRIDE' in globals() else None
    review_override = pd.DataFrame(SALES_REVIEWS_OVERRIDE, columns=REVIEW_COLUMNS) if 'SALES_REVIEWS_OVERRIDE' in globals() else None
    if 'CYCLE_REPORTS_OVERRIDE' in globals():
        chosen_database = Path(globals().get('DAILY_DATABASE_OVERRIDE', globals().get('DATABASE_OVERRIDE', tracking['database'])))
        tracking_days, tracking_rows = read_cycle_history(CYCLE_REPORTS_OVERRIDE, chosen_database, as_of=TRACKING_AS_OF)
        chosen_timezone = tracking_days.timezone.iloc[0] if not tracking_days.empty else tracking['timezone']
        tracking_tables = daily_tables(tracking_days, tracking_rows, as_of=TRACKING_AS_OF,
            timezone_name=chosen_timezone, followup_limit=tracking['followup_limit'], checks=check_override, reviews=review_override)
        print('Explicit cycle selection:', CYCLE_REPORTS_OVERRIDE, '| Database:', chosen_database)
    else:
        tracking_tables = tracking_report(tracking, as_of=TRACKING_AS_OF, checks=check_override, reviews=review_override)
        print('Daily database:', tracking['database'], '| Configured queries:', len(tracking['queries']))
    inventory_daily = tracking_tables['daily_inventory']
    vehicle_daily = tracking_tables['vehicle_observations']
    detail_followups = tracking_tables['detail_followups']
    saved_check_evidence = tracking_tables['listing_checks']
    selected_review_evidence = tracking_tables['selected_reviews']
    operating_candidates = tracking_tables['sale_candidates']
    print('Evidence cutoff:', TRACKING_AS_OF)
else:
    print('Operating register view explicitly skipped.')

Daily database: C:\Users\Sean\VscProjects\researchOS\vehicle\data\analysis\carvana_daily\history.sqlite | Configured queries: 7
Evidence cutoff: 2026-09-09T02:53:55.255294+00:00


## 1. What population and observation windows are we comparing?

The config fixes the filters. The register selects actual `cycle.json` reports;
the reader verifies their retained evidence and opens SQLite read-only. A second
complete, consecutive date is needed for a one-day comparison. Missing dates and
partial collections remain visible.

`AS_OF_OVERRIDE` can select a historical evidence cutoff. `CYCLE_REPORTS_OVERRIDE`
and `DAILY_DATABASE_OVERRIDE` deliberately select other retained cycles/database;
their checks/reviews default to empty unless explicitly overridden. Older morning
experiments never become daily snapshots by changing a label.

In [3]:
from vehicle_tracker.cycles import read_cycle_history
from vehicle_tracker.events import vin_events, daily_counts
if 'CYCLE_REPORTS_OVERRIDE' in globals():
    CYCLE_REPORTS = CYCLE_REPORTS_OVERRIDE
    DAILY_DATABASE = Path(globals().get('DAILY_DATABASE_OVERRIDE', globals().get('DATABASE_OVERRIDE', globals().get('DATABASE', Path('vehicle/data/analysis/carvana_daily/history.sqlite')))))
    AS_OF = globals().get('AS_OF_OVERRIDE', '2026-09-08T13:00:00Z')
else:
    daily_settings = tracking_settings(TRACKING_CONFIG_PATH)
    CYCLE_REPORTS = [row['path'] for row in registered_cycles(daily_settings)]
    DAILY_DATABASE = daily_settings['database']
    AS_OF = TRACKING_AS_OF

daily_cycles = daily_observations = daily_source_rows = pd.DataFrame()
daily_events = daily_summary = daily_timeline = daily_identity_join = pd.DataFrame()
if CYCLE_REPORTS:
    daily_cycles, daily_observations = read_cycle_history([Path(path) for path in CYCLE_REPORTS], DAILY_DATABASE, as_of=AS_OF)
if daily_cycles.empty:
    print('NO DAILY CYCLES: no retained cycles available for this selection/cutoff. Morning runs are not daily cycles.')
else:
    collection_windows = daily_cycles[['cycle_date', 'timezone', 'coverage_complete', 'coverage_reason']].copy()
    for field in ['window_start', 'window_end']:
        collection_windows[field + '_local'] = daily_cycles.apply(
            lambda row: pd.Timestamp(row[field]).tz_convert(row.timezone).isoformat(), axis=1)
    display(collection_windows)
    print('Scope ID:', daily_cycles.scope_id.unique().tolist())
print('Daily evidence cutoff:', AS_OF, 'Current-code reanalysis; preserve code/config hashes for a vintage.')

,cycle_date,timezone,coverage_complete,coverage_reason,window_start_local,window_end_local
0,2026-09-08,America/New_York,True,Complete requested daily scope,2026-09-08T21:44:11.074146-04:00,2026-09-08T21:45:40.704618-04:00


Scope ID: ['4a8ee109d5ada141f87906d073c725319b96c248fbcad10dab9f4ca3e78d50bb']
Daily evidence cutoff: 2026-09-09T02:53:55.255294+00:00 Current-code reanalysis; preserve code/config hashes for a vintage.


## 2. What did each snapshot contain?

`observed_vins` counts what was actually seen; `inventory_count` is withheld on an
incomplete day. Missing is not zero. `pending_true` counts the native API flag and
`pending_unknown` counts missing flags. **Not pending does not mean ready to buy:**
the page study below found pre-order vehicles in that group.

The first few vehicle rows show the raw ingredients. Their source paths link each
observation back to a retained response. The main sales-estimate column stays missing.

In [4]:
if tracking_tables and not inventory_daily.empty:
    display(inventory_daily[['cycle_date', 'coverage_status', 'observed_vins', 'inventory_count',
        'pending_true', 'pending_unknown', 'average_asking_price_usd', 'estimated_sales']])
    display(vehicle_daily[['cycle_date', 'vin', 'listing_id', 'observed_at_utc',
        'purchase_pending', 'asking_price_usd', 'source_path']].head(5))
    problems = inventory_daily.loc[inventory_daily.coverage_status.ne('complete')]
    if not problems.empty:
        display(problems[['cycle_date', 'coverage_status', 'coverage_reason', 'analysis_error']])
else:
    print('No operating daily table for this selection. No counts are assumed.')

,cycle_date,coverage_status,observed_vins,inventory_count,pending_true,pending_unknown,average_asking_price_usd,estimated_sales
0,2026-09-08,complete,674,674,215,0,28219.376855,<NA>


,cycle_date,vin,listing_id,observed_at_utc,purchase_pending,asking_price_usd,source_path
0,2026-09-08,5YJ3E1EA1TF180441,4661065,2026-09-09T01:45:40.704618+00:00,0,40990.0,C:\Users\Sean\VscProjects\researchOS\vehicle\d...
1,2026-09-08,5YJ3E1EB8TF116843,4729844,2026-09-09T01:45:40.704618+00:00,0,47590.0,C:\Users\Sean\VscProjects\researchOS\vehicle\d...
2,2026-09-08,5YJ3E1EA2TF142040,4653422,2026-09-09T01:45:40.704618+00:00,0,41590.0,C:\Users\Sean\VscProjects\researchOS\vehicle\d...
3,2026-09-08,5YJ3E1EA7TF134113,4627958,2026-09-09T01:45:40.704618+00:00,0,40990.0,C:\Users\Sean\VscProjects\researchOS\vehicle\d...
4,2026-09-08,5YJ3E1EA8TF183854,4671434,2026-09-09T01:45:40.704618+00:00,0,43590.0,C:\Users\Sean\VscProjects\researchOS\vehicle\d...


## 3. Which VINs appeared, disappeared, remained, or changed pending status?

The visible outer merge below matches **retailer + VIN**. `left_only` means seen
before but absent after; `right_only` means new to the comparison; `both` means
observed on both days. A changed listing ID remains visible as a relisting diagnostic.

We compare the last two selected cycles only if both are complete, consecutive,
and pass the existing identity/scope checks. We never skip a failed day to manufacture
a daily change. Pending comparisons require known values on both sides.
The collection windows are not instantaneous snapshots.

The event history retains reappearances, coverage gaps, and last-seen evidence.
Three consecutive complete absent days create a review candidate, not a sale.

In [5]:
daily_comparison_allowed = False
daily_analysis_error = None
daily_change_counts = pd.DataFrame()
if not daily_cycles.empty:
    try:
        daily_events = vin_events(daily_cycles, daily_observations, absence_days=3)
        daily_summary = daily_counts(daily_cycles, daily_events)
    except ValueError as error:
        daily_analysis_error = str(error)
        print('BLOCKED DAILY ANALYSIS:', daily_analysis_error)
    if daily_analysis_error is None:
        daily_source_rows = daily_observations.merge(daily_cycles[['cycle_id', 'cycle_date', 'coverage_complete']],
            on='cycle_id', validate='many_to_one')
        VIN_KEY = globals().get('VIN_KEY_OVERRIDE', tuple(daily_events[['retailer', 'vin']].iloc[0])
            if not daily_events.empty else (None, None))
        daily_timeline = daily_events.loc[daily_events.retailer.eq(VIN_KEY[0]) & daily_events.vin.eq(VIN_KEY[1])]
        last_two = daily_cycles.sort_values('cycle_date').tail(2)
        daily_comparison_allowed = (len(last_two) == 2 and last_two.coverage_complete.all()
            and pd.to_datetime(last_two.cycle_date).diff().iloc[-1] == pd.Timedelta(days=1))
        if daily_comparison_allowed:
            before = daily_observations.loc[daily_observations.cycle_id.eq(last_two.iloc[0].cycle_id)]
            after = daily_observations.loc[daily_observations.cycle_id.eq(last_two.iloc[1].cycle_id)]
            fields = ['retailer', 'vin', 'listing_id', 'purchase_pending', 'asking_price_usd']
            display(before[fields].head(3))
            display(after[fields].head(3))
            daily_identity_join = before.merge(after, on=['retailer', 'vin'], how='outer',
                suffixes=('_before', '_after'), indicator=True, validate='one_to_one')
            matched = daily_identity_join['_merge'].eq('both')
            known_pending = daily_identity_join.purchase_pending_before.notna() & daily_identity_join.purchase_pending_after.notna()
            daily_identity_join['pending_changed'] = daily_identity_join.purchase_pending_before.ne(
                daily_identity_join.purchase_pending_after).astype('boolean').where(matched & known_pending)
            daily_identity_join['visible_asking_price_change_usd'] = (daily_identity_join.asking_price_usd_after
                - daily_identity_join.asking_price_usd_before).where(matched)
            daily_change_counts = daily_identity_join.groupby('_merge', observed=False).size().rename('VINs').to_frame()
            print('Compared dates:', last_two.cycle_date.tolist())
            display(daily_change_counts)
            print('Known pending changes:', int(daily_identity_join.pending_changed.eq(True).sum()),
                  '| Matched VINs with unknown pending comparison:', int((matched & ~known_pending).sum()))
            display(daily_identity_join[['vin', 'listing_id_before', 'listing_id_after', '_merge',
                'purchase_pending_before', 'purchase_pending_after', 'pending_changed',
                'visible_asking_price_change_usd']].head(12))
        else:
            print('BLOCKED DAILY JOIN: need two complete consecutive selected dates. Baseline/gap/partial is not zero change.')

BLOCKED DAILY JOIN: need two complete consecutive selected dates. Baseline/gap/partial is not zero change.


## 4. What did checking the actual vehicle pages show?

These are saved **manual browser observations**, not automatic page requests.
We matched each page's VIN and listing URL before recording the result. The source
contains the exact status wording, its location, identity evidence, and check time.

| Recorded status | What it establishes |
| --- | --- |
| `available` | The loaded page offered purchase, with a Get Started control |
| `pending` | The page displayed a purchase-in-progress or hold state |
| `sold_label` | An explicit status said this listing was sold; delivery date remains unknown |
| `unavailable` | No longer available, without an explicit sold status |
| `access_blocked` | The page could not be accessed; vehicle status is unestablished |
| `unknown` | No other category was established; read the native wording and note |

The September 8 study selected the first two pending VINs and first three
non-pending VINs in VIN order: five controls from one baseline, no missing-VIN
cohort yet. Two pre-order pages are retained as `unknown`, with “Pre-order now”
and “Inspection in progress” preserved; we did not add a new status system.

All five pages contained generic wording about equipment “as originally sold.”
That is not the vehicle's sale status. No explicit Sold status was observed.
This convenience sample cannot estimate accuracy, sales, or reappearance rates.

The merge uses retailer + VIN + listing ID. It attaches the latest inventory
observation at or before each physical page check. Different source times and
browser context can explain differences; they are not proven orders/cancellations.

In [6]:
check_context = page_check_counts = check_history_visible = later_inventory = pd.DataFrame()
if tracking_tables and not daily_source_rows.empty:
    if 'CHECKS_OVERRIDE' in globals():
        check_history = pd.DataFrame(CHECKS_OVERRIDE, columns=CHECK_COLUMNS)
    elif 'CYCLE_REPORTS_OVERRIDE' in globals():
        check_history = pd.DataFrame(columns=CHECK_COLUMNS)
    else:
        check_history = read_records(tracking['checks'], CHECK_COLUMNS)
    check_history = validate_checks(check_history)
    check_history = check_history.loc[pd.to_datetime(check_history.available_at, utc=True, format='ISO8601').le(pd.Timestamp(TRACKING_AS_OF))]
    # Keep the latest known correction to each physical check; retain separate check times.
    check_history_visible = check_history.sort_values('available_at').drop_duplicates(
        ['retailer', 'vin', 'listing_id', 'checked_at'], keep='last')
    if saved_check_evidence.empty:
        print('No saved page checks at this cutoff. That does not mean zero sales.')
    else:
        identity = ['retailer', 'vin', 'listing_id']
        inventory_context = daily_source_rows[identity + ['cycle_date', 'observed_at_utc', 'purchase_pending']]
        check_context = saved_check_evidence.merge(inventory_context, on=identity, validate='one_to_many')
        check_context = check_context.loc[pd.to_datetime(check_context.observed_at_utc, utc=True, format='ISO8601').le(
            pd.to_datetime(check_context.checked_at, utc=True, format='ISO8601'))]
        check_context = check_context.sort_values('observed_at_utc').drop_duplicates('check_id', keep='last')
        display(check_context[['vin', 'listing_id', 'observed_at_utc', 'purchase_pending',
            'checked_at', 'observed_status', 'native_text']])
        page_check_counts = saved_check_evidence.groupby('observed_status').size().rename('checked_listings').to_frame()
        display(page_check_counts)
        print('Counts cover checked listings only, not daily sales or the full population.')
        display(check_context[['listing_id', 'available_at', 'source', 'note']])
        repeated = check_history_visible.duplicated(identity, keep=False)
        if repeated.any():
            display(check_history_visible.loc[repeated, identity + ['checked_at', 'observed_status', 'native_text']]
                .sort_values(identity + ['checked_at']))
        else:
            print('No repeated physical page checks yet; no page-status trajectory can be measured.')
        # Same VIN can return under a new listing ID. Keep the two IDs distinct.
        later_inventory = saved_check_evidence[identity + ['checked_at']].rename(columns={'listing_id': 'checked_listing_id'}).merge(
            daily_events, on=['retailer', 'vin'], validate='many_to_many')
        later_inventory = later_inventory.loc[pd.to_datetime(later_inventory.window_start, utc=True, format='ISO8601').gt(
            pd.to_datetime(later_inventory.checked_at, utc=True, format='ISO8601'))]
        if later_inventory.empty:
            print('No later inventory cycle after these checks yet; reappearance follow-up is unavailable.')
        else:
            display(later_inventory[['vin', 'checked_listing_id', 'checked_at', 'cycle_date',
                'listing_id', 'event_type', 'observed_in_cycle', 'reappeared_after_absence', 'coverage_complete']].head(20))
elif tracking_tables:
    print('No valid inventory evidence for page-check comparison at this cutoff.')

,vin,listing_id,observed_at_utc,purchase_pending,checked_at,observed_status,native_text
3,5YJ3E1EA0MF020584,4715778,2026-09-09T01:45:10.829512+00:00,0,2026-09-09T02:38:47.406000+00:00,pending,On Hold\nAnother customer started purchasing t...
1,5YJ3E1EA0LF784711,4671091,2026-09-09T01:45:23.494370+00:00,0,2026-09-09T02:38:33.600000+00:00,unknown,Pre-order now\nVehicle is almost ready\nInspec...
2,5YJ3E1EA0LF793165,4460674,2026-09-09T01:45:23.494370+00:00,0,2026-09-09T02:38:41.503000+00:00,unknown,Pre-order now\nVehicle is almost ready\nInspec...
0,5YJ3E1EA0LF632279,4681095,2026-09-09T01:45:28.912560+00:00,1,2026-09-09T02:38:24.322000+00:00,available,"Get Started\nGet it Mon , Sep 21"
4,5YJ3E1EA0LF611240,4640427,2026-09-09T01:45:28.912560+00:00,1,2026-09-09T02:38:52.480000+00:00,pending,Purchase in progress\nAnother customer started...


,checked_listings
observed_status,
available,1
pending,2
unknown,2


Counts cover checked listings only, not daily sales or the full population.


,listing_id,available_at,source,note
3,4715778,2026-09-09T02:40:22.105430+00:00,C:\Users\Sean\VscProjects\researchOS\vehicle\d...,https://www.carvana.com/vehicle/4715778 | The ...
1,4671091,2026-09-09T02:40:22.105430+00:00,C:\Users\Sean\VscProjects\researchOS\vehicle\d...,https://www.carvana.com/vehicle/4671091 | The ...
2,4460674,2026-09-09T02:40:22.105430+00:00,C:\Users\Sean\VscProjects\researchOS\vehicle\d...,https://www.carvana.com/vehicle/4460674 | The ...
0,4681095,2026-09-09T02:40:22.105430+00:00,C:\Users\Sean\VscProjects\researchOS\vehicle\d...,https://www.carvana.com/vehicle/4681095 | The ...
4,4640427,2026-09-09T02:40:22.105430+00:00,C:\Users\Sean\VscProjects\researchOS\vehicle\d...,https://www.carvana.com/vehicle/4640427 | The ...


No repeated physical page checks yet; no page-status trajectory can be measured.
No later inventory cycle after these checks yet; reappearance follow-up is unavailable.


### Candidates and the next page checks

The existing queue retains first absences, native changes and unresolved checks;
at most 20 are selected. It does not browse automatically. A recently checked
listing can be `up_to_date`; unresolved checks become due after 48 hours.
That is an operating interval, not a sales rule.

The existing three-day candidate rule is unchanged. `reviewed_sales_with_known_date`
counts only explicit dated analyst confirmations, not all sales. A saved Sold
label does not automatically create a confirmation or transaction date. Old checks
and reviews remain in their histories; the report uses only evidence available
by its cutoff. See [the recording guide](../docs/listing_checks.md) for commands.

In [7]:
from vehicle_tracker.sales import sale_candidates, REVIEW_COLUMNS
from vehicle_tracker.checks import select_reviews
if 'SALES_REVIEWS_OVERRIDE' in globals():
    review_history = pd.DataFrame(SALES_REVIEWS_OVERRIDE, columns=REVIEW_COLUMNS)
elif 'CYCLE_REPORTS_OVERRIDE' in globals():
    review_history = pd.DataFrame(columns=REVIEW_COLUMNS)
else:
    _, review_history = review_inputs(daily_settings)
SALES_REVIEWS = select_reviews(review_history, as_of=AS_OF)
sale_candidate_rows = sale_daily_report = pd.DataFrame()
if globals().get('daily_analysis_error') is None:
    sale_candidate_rows, sale_daily_report = sale_candidates(
        daily_cycles, daily_observations, as_of=AS_OF, absence_days=3, reviews=SALES_REVIEWS)
    if not sale_candidate_rows.empty:
        display(sale_candidate_rows[['vin', 'listing_id', 'first_absent_date', 'detected_date',
            'followup_state', 'review_outcome', 'sale_date', 'review_needs_followup']])
    else:
        print('No qualifying reviewed candidate to display. Sales remain unknown.')
if globals().get('tracking_tables') and not detail_followups.empty:
    display(detail_followups[['vin', 'listing_id', 'followup_reason', 'detail_status',
        'queue_state', 'selected_for_check', 'listing_url']].head(20))

No qualifying reviewed candidate to display. Sales remain unknown.


,vin,listing_id,followup_reason,detail_status,queue_state,selected_for_check,listing_url
0,5YJ3E1EA0LF784711,4671091,unresolved_check,unknown,up_to_date,False,https://www.carvana.com/vehicle/4671091
1,5YJ3E1EA0LF793165,4460674,unresolved_check,unknown,up_to_date,False,https://www.carvana.com/vehicle/4460674
2,5YJ3E1EA0MF020584,4715778,unresolved_check,pending,up_to_date,False,https://www.carvana.com/vehicle/4715778
3,5YJ3E1EA0LF611240,4640427,unresolved_check,pending,up_to_date,False,https://www.carvana.com/vehicle/4640427


## 5. What can we conclude, and what happens next?

The September 8 study established that these five pages were readable in ordinary
Chrome and showed different purchase states. One API-pending vehicle offered
purchase at the later page check; another API-non-pending vehicle showed a hold.
Those are cross-source observations, not proven cancellations or new orders.
The two pre-order pages show why the native pending flag alone is insufficient.

**Sold-label reliability is still unmeasured.** We have no disappeared-VIN cohort,
no explicit Sold status in this study, and no later daily follow-up. Ordinary
browser success does not demonstrate reliable unattended page collection.
The decision for this milestone is to establish consecutive collection and
status coverage first, before choosing a disappearance-to-sales conversion.

Next, collect the unchanged pilot on September 9 at approximately 21:45 New York
time, then rerun this notebook. If that date has passed, use the actual current
date and retain any gap. Investigate first disappearances and pending changes,
and revisit the saved checks when there is new evidence. An explicit Sold label
would start a follow-up study of persistence/reappearance and timing; its check
date would still not establish a delivery date.

Run outside the notebook, from the repository root:

```powershell
.\.venv\Scripts\python.exe -B vehicle/scripts/run_carvana_daily.py
.\.venv\Scripts\python.exe -B vehicle/scripts/run_carvana_daily.py --live
```

The first command previews. The second deliberately collects/imports/exports
the configured pilot, at most 120 requests / 20 minutes, and refuses a second
registered run for the same local date. `--refresh` writes a new export from
existing evidence only. Opening this notebook writes nothing.

[Study evidence and limitations](../docs/status_validation_20260908.md) ·
[Daily operating guide](../docs/daily_inventory.md) ·
[Earlier intraday reference](21_carvana_intraday_reference.ipynb)

## Optional learning example: one invented VIN over eight days

This example runs even when no real daily cycles are selected. Its data are **synthetic, in memory only**, and never mixed into the real results.

| Day | Invented observation | Interpretation |
| --- | --- | --- |
| 1 | Old listing ID, $20,000, pending | Starting observation |
| 2 | Same VIN, new listing ID, $19,500, not pending | Relisting diagnostic; asking price fell $500 |
| 3 | No collection | Gap, not absence evidence |
| 4 | Incomplete collection | Absence cannot be assessed |
| 5-7 | Complete collections without the VIN | Three-day absence threshold reached on day 7 |
| 8 | VIN returns | Reappearance |

The two-day threshold is reached on day 6, the three-day threshold on day 7, and the seven-day threshold is not reached. No event establishes a sale. Read the displayed source tables first, then the calculated event/count tables.

In [8]:
print('SYNTHETIC IN-MEMORY EXAMPLE ONLY: invented dates and VIN; no collection or sales evidence.')
synthetic_days = ['01', '02', '04', '05', '06', '07', '08']
synthetic_cycles = pd.DataFrame([dict(cycle_id='synthetic-' + day, cycle_date='2026-09-' + day, timezone='UTC', scope_id='synthetic-only',
    window_start=f'2026-09-{day}T10:00:00Z', window_end=f'2026-09-{day}T11:00:00Z', available_at=f'2026-09-{day}T11:05:00Z',
    coverage_complete=day != '04', coverage_reason='Synthetic incomplete day' if day == '04' else 'Synthetic complete day') for day in synthetic_days])
synthetic_observations = pd.DataFrame([dict(cycle_id='synthetic-' + day, retailer='carvana', vin='SYNTHETIC-VIN-A', listing_id=listing,
    observed_at_utc=f'2026-09-{day}T10:30:00Z', capture_id='synthetic-capture-' + day, run_id='synthetic-run-' + day,
    source_url='synthetic://example/' + day, listing_url='synthetic://listing/' + listing,
    asking_price_usd=price, purchase_pending=pending, vehicle_lock_type=0)
    for day, listing, price, pending in [('01', 'old', 20000, True), ('02', 'new', 19500, False), ('08', 'new', 19500, False)]])
synthetic_events = vin_events(synthetic_cycles, synthetic_observations, absence_days=3)
synthetic_summary = daily_counts(synthetic_cycles, synthetic_events)
display(synthetic_cycles[['cycle_date', 'coverage_complete', 'coverage_reason']])
display(synthetic_observations[['cycle_id', 'vin', 'listing_id', 'asking_price_usd', 'purchase_pending']])
display(synthetic_events[['cycle_date', 'vin', 'listing_id', 'event_type', 'observed_in_cycle', 'absence_streak', 'timing_uncertain', 'asking_price_change_usd', 'estimated_sales']])
display(synthetic_summary[['cycle_date', 'skipped_days_before', 'pending_cleared', 'first_absence', 'persistent_absence', 'estimated_sales']])


SYNTHETIC IN-MEMORY EXAMPLE ONLY: invented dates and VIN; no collection or sales evidence.


,cycle_date,coverage_complete,coverage_reason
0,2026-09-01,True,Synthetic complete day
1,2026-09-02,True,Synthetic complete day
2,2026-09-04,False,Synthetic incomplete day
3,2026-09-05,True,Synthetic complete day
4,2026-09-06,True,Synthetic complete day
5,2026-09-07,True,Synthetic complete day
6,2026-09-08,True,Synthetic complete day


,cycle_id,vin,listing_id,asking_price_usd,purchase_pending
0,synthetic-01,SYNTHETIC-VIN-A,old,20000,True
1,synthetic-02,SYNTHETIC-VIN-A,new,19500,False
2,synthetic-08,SYNTHETIC-VIN-A,new,19500,False


,cycle_date,vin,listing_id,event_type,observed_in_cycle,absence_streak,timing_uncertain,asking_price_change_usd,estimated_sales
0,2026-09-01,SYNTHETIC-VIN-A,old,first_observed,True,0,True,<NA>,<NA>
1,2026-09-02,SYNTHETIC-VIN-A,new,relisted,True,0,False,-500,<NA>
2,2026-09-04,SYNTHETIC-VIN-A,new,absence_unassessable,False,0,True,<NA>,<NA>
3,2026-09-05,SYNTHETIC-VIN-A,new,first_absence,False,1,True,<NA>,<NA>
4,2026-09-06,SYNTHETIC-VIN-A,new,continued_absence,False,2,True,<NA>,<NA>
5,2026-09-07,SYNTHETIC-VIN-A,new,persistent_absence,False,3,True,<NA>,<NA>
6,2026-09-08,SYNTHETIC-VIN-A,new,reappeared,True,0,True,0,<NA>


,cycle_date,skipped_days_before,pending_cleared,first_absence,persistent_absence,estimated_sales
0,2026-09-01,0,0,0,0,<NA>
1,2026-09-02,0,1,0,0,<NA>
2,2026-09-04,1,0,<NA>,<NA>,<NA>
3,2026-09-05,0,0,1,0,<NA>
4,2026-09-06,0,0,0,0,<NA>
5,2026-09-07,0,0,0,1,<NA>
6,2026-09-08,0,0,0,0,<NA>


### The same invented VIN becomes a candidate, then reappears

At the end of synthetic day 7, the three-day absence rule has produced one candidate. At the end of day 8, the same candidate is marked `reappeared`. Its original detection date does not change, no second candidate is added, and no sale is inferred. The two tables show exactly what was knowable at those different cutoffs.

In [9]:
print('SYNTHETIC SALE-CANDIDATE EXAMPLE ONLY; no reviewed outcome or sale is invented.')
example_candidates_before, _ = sale_candidates(synthetic_cycles, synthetic_observations,
    as_of='2026-09-07T23:00:00Z', absence_days=3)
example_candidates_after, example_candidate_daily = sale_candidates(synthetic_cycles, synthetic_observations,
    as_of='2026-09-08T23:00:00Z', absence_days=3)
for label, frame in [('Known on day 7', example_candidates_before), ('Known on day 8', example_candidates_after)]:
    print(label)
    display(frame[['vin', 'detected_date', 'detected_available_at', 'followup_state',
                   'reappeared_date', 'review_outcome', 'sale_date']])

SYNTHETIC SALE-CANDIDATE EXAMPLE ONLY; no reviewed outcome or sale is invented.
Known on day 7


,vin,detected_date,detected_available_at,followup_state,reappeared_date,review_outcome,sale_date
0,SYNTHETIC-VIN-A,2026-09-07,2026-09-07T11:05:00+00:00,still_absent,None,unreviewed,None


Known on day 8


,vin,detected_date,detected_available_at,followup_state,reappeared_date,review_outcome,sale_date
0,SYNTHETIC-VIN-A,2026-09-07,2026-09-07T11:05:00+00:00,reappeared,2026-09-08,unreviewed,None
